# Vision-Driven Computer Use

`06_BrowserAgent_Computer_Use_Applied.ipynb` teaches the shape of a computer-use loop, but
is explicit that it is a simulation: its `screenshot_text()` returns a *text description* of
the page. The agent never sees anything.

This notebook closes that. The agent receives an actual **PNG image** of a UI, and has to
work out from pixels where things are and what to click. Nothing is described to it in
words.

That change is not cosmetic. When the observation is text, someone has already done the
hard part — parsing the interface into named elements. When it is pixels, the model has to
do that itself, and everything that makes computer use unreliable shows up.

## Learning objectives

1. Build an act–observe loop whose observation is an image, not a description.
2. Send a screenshot to a vision model and parse a structured action back.
3. Use it as a **frontend testing** agent: give it a goal and let it find the bug.
4. Name why coordinate-based control is brittle, and what production systems do instead.

## Where this fits

- `06_BrowserAgent_Computer_Use_Applied.ipynb` — the same loop with text observations.
  Read it first; this is the pixel version.
- `07_Hosted_vs_Client_Side_Tools.ipynb` — OpenAI's `computer_use_preview` is the hosted
  version of everything below, with the trade-offs that notebook describes.

## Dependencies, deliberately none new

The UI is rendered with **Pillow**, which is already installed, and the screenshots are real
PNGs. No Playwright, no Selenium, no headless Chromium download — none of which are
dependencies of this repo.

What that costs: the GUI is drawn rather than browsed. What it keeps: real pixels, real
vision, real coordinates, and a loop that behaves the way a browser-driven one does. The
final section explains exactly what changes when you swap the renderer for a real browser.

## Prerequisites

`OPENAI_API_KEY` in the project-root `.env`. Vision calls cost more than text — roughly
`steps × screenshots`, a few cents.

In [ ]:
# ============ SETUP ============
import base64
import io
import json

from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image, ImageDraw, ImageFont

load_dotenv()
client = OpenAI()

VISION_MODEL = "gpt-4o-mini"   # must accept image input
MAX_STEPS = 6                  # hard cap: a confused agent will loop forever otherwise

def _font(size: int):
    for path in ("/System/Library/Fonts/Supplemental/Arial.ttf",
                 "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                 "C:/Windows/Fonts/arial.ttf"):
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            continue
    return ImageFont.load_default()

## 1. A UI made of pixels

`Screen` draws a small checkout page and tracks its own state. The important detail is what
it does **not** expose: there is no `get_elements()`, no accessibility tree, no DOM. The only
way out is `screenshot()`, which returns a PNG.

Clicks arrive as `(x, y)`. The screen decides what, if anything, was hit — exactly as a real
GUI does.

In [ ]:
# ============ THE UI ============
class Screen:
    """A drawn checkout page. Observable only as pixels; clickable only by coordinate."""

    W, H = 520, 360

    def __init__(self, coupon_button_broken: bool = False):
        self.qty = 1
        self.coupon_applied = False
        self.placed = False
        self.coupon_button_broken = coupon_button_broken   # the bug, for section 4
        self.click_log: list[tuple[int, int, str]] = []
        # name -> (x1, y1, x2, y2). The agent never sees this.
        self.boxes = {
            "qty_plus":   (250, 96, 286, 130),
            "qty_minus":  (196, 96, 232, 130),
            "coupon":     (40, 190, 210, 228),
            "place":      (40, 262, 260, 306),
        }

    def _hit(self, x: int, y: int) -> str | None:
        for name, (x1, y1, x2, y2) in self.boxes.items():
            if x1 <= x <= x2 and y1 <= y <= y2:
                return name
        return None

    def click(self, x: int, y: int) -> str:
        target = self._hit(x, y)
        self.click_log.append((x, y, target or "MISS"))
        if target == "qty_plus":
            self.qty += 1
        elif target == "qty_minus":
            self.qty = max(1, self.qty - 1)
        elif target == "coupon":
            if not self.coupon_button_broken:
                self.coupon_applied = True
        elif target == "place":
            self.placed = True
        return target or "MISS"

    def screenshot(self) -> Image.Image:
        img = Image.new("RGB", (self.W, self.H), "#f4f4f6")
        d = ImageDraw.Draw(img)
        big, mid, small = _font(20), _font(16), _font(13)

        d.text((40, 30), "Checkout", fill="#111", font=big)
        d.text((40, 70), "Wireless Mouse", fill="#333", font=mid)

        # quantity stepper
        d.rectangle(self.boxes["qty_minus"], outline="#555", width=2, fill="white")
        d.text((208, 103), "-", fill="#111", font=mid)
        d.text((240, 103), str(self.qty), fill="#111", font=mid)
        d.rectangle(self.boxes["qty_plus"], outline="#555", width=2, fill="white")
        d.text((262, 103), "+", fill="#111", font=mid)

        # coupon
        fill = "#cdebd6" if self.coupon_applied else "white"
        d.rectangle(self.boxes["coupon"], outline="#555", width=2, fill=fill)
        label = "Coupon applied" if self.coupon_applied else "Apply coupon"
        d.text((54, 202), label, fill="#111", font=mid)

        # place order
        d.rectangle(self.boxes["place"], outline="#1a5", width=3,
                    fill="#1a5" if not self.placed else "#888")
        d.text((60, 277), "Order placed" if self.placed else "Place order",
               fill="white", font=mid)

        d.text((40, 326), f"total: ${12.50 * self.qty:.2f}", fill="#333", font=small)
        return img


def to_data_url(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


demo = Screen()
print(f"  screenshot: {demo.screenshot().size[0]}x{demo.screenshot().size[1]} PNG")
demo.screenshot()

## 2. The agent sees pixels

One call: the screenshot goes in as an image, a JSON action comes back. The prompt gives the
model the image dimensions and the action vocabulary — and nothing about what is on screen.

Note the action space is deliberately tiny: `click(x, y)` or `done`. Every extra verb is
another thing the model can get wrong.

In [ ]:
# ============ ONE VISION STEP ============
SYSTEM = (
    "You operate a GUI by looking at screenshots. You are given a PNG of the current screen "
    f"({Screen.W}x{Screen.H} pixels, origin top-left). Decide the single next action.\n"
    "Reply ONLY with JSON, no prose, no code fences:\n"
    '  {"action": "click", "x": <int>, "y": <int>, "why": "<short>"}\n'
    '  {"action": "done", "why": "<short>"}\n'
    "Click the CENTRE of the control you intend to press."
)


def decide(img: Image.Image, goal: str, history: list[str]) -> dict:
    past = ("\n".join(f"- {h}" for h in history)) or "- (nothing yet)"
    reply = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": [
                {"type": "text",
                 "text": f"Goal: {goal}\n\nActions so far:\n{past}\n\nNext action?"},
                {"type": "image_url", "image_url": {"url": to_data_url(img)}},
            ]},
        ],
    )
    raw = (reply.choices[0].message.content or "").strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"action": "done", "why": f"unparseable reply: {raw[:60]}"}

## 3. The loop

Screenshot → decide → act → screenshot. The agent is told only whether its click **hit
something or missed**, never what the thing was called. Missing is information too, and a
capable agent should correct after one.

In [ ]:
# ============ ACT-OBSERVE LOOP ============
def run(goal: str, screen: Screen, max_steps: int = MAX_STEPS, verbose: bool = True):
    history: list[str] = []
    for step in range(1, max_steps + 1):
        action = decide(screen.screenshot(), goal, history)
        if action.get("action") == "done":
            if verbose:
                print(f"  {step}. done — {action.get('why','')}")
            break
        x, y = int(action.get("x", -1)), int(action.get("y", -1))
        hit = screen.click(x, y)
        note = f"clicked ({x},{y}) -> {hit}"
        history.append(note)
        if verbose:
            print(f"  {step}. {note}   [{action.get('why','')[:44]}]")
    return screen


print("GOAL: set quantity to 3, apply the coupon, then place the order\n")
screen = run("Set the quantity to 3, apply the coupon, then place the order.", Screen())

print(f"\n  qty={screen.qty}  coupon={screen.coupon_applied}  placed={screen.placed}")
misses = sum(1 for *_, t in screen.click_log if t == "MISS")
print(f"  clicks: {len(screen.click_log)} ({misses} missed)")

### Discussion of the output

Watch the `MISS` count. That number is the entire reliability story of coordinate-based
computer use, and it is why this is the least dependable tool-use pattern in the repo.

A miss is not a crash. The screen simply does not change, the agent takes another
screenshot, and if it does not notice it will click the same wrong spot again. That is the
characteristic failure: **not an error, a silent no-op loop.** The step cap is what turns it
into a bounded failure instead of an unbounded bill.

You will also see the model describe controls accurately while mislocating them by tens of
pixels. Recognising *what* is on screen and knowing *where* it is are different abilities,
and vision models are markedly better at the first.

## 4. As a frontend testing agent

This is OpenAI's own example use case for computer use, and it is a better fit than
general automation, because a test has something the open-ended case lacks: **a
verifiable end state.**

Below, the coupon button is wired to do nothing — a plausible frontend regression. The
agent is not told. We give it the same goal and then assert on the result.

In [ ]:
# ============ THE AGENT MEETS A BROKEN BUTTON ============
broken = Screen(coupon_button_broken=True)
print("GOAL (against a build where the coupon button is broken)\n")
run("Set the quantity to 3, apply the coupon, then place the order.", broken)

print("\n  --- assertions ---")
checks = {
    "quantity is 3":   broken.qty == 3,
    "coupon applied":  broken.coupon_applied,
    "order placed":    broken.placed,
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

coupon_clicks = [c for c in broken.click_log if c[2] == "coupon"]
if coupon_clicks and not broken.coupon_applied:
    print(f"\n  diagnosis: the agent hit the coupon control {len(coupon_clicks)}x and state "
          f"never changed -> the control is broken, not the agent.")

### Why that diagnosis line matters

A failing assertion alone is nearly useless: *"coupon not applied"* could mean the button is
broken, or that the agent never found it. Those need different people to fix them.

The click log separates them. **Hit the control and nothing happened → your bug. Never hit
it → the agent's problem.** Any computer-use test worth running needs that distinction built
in, because without it every failure lands back on whoever owns the agent.

## 5. What changes with a real browser

Everything above is a genuine vision loop — real PNGs, real pixel coordinates, real
misclicks. What a real browser would change:

| | Here | Playwright / Selenium |
|---|---|---|
| Rendering | Pillow draws it | A real engine, real fonts, real layout |
| What can break | Only what I coded | Scroll position, overlays, timing, focus, animation |
| Element access | Coordinates only | Coordinates **and** selectors / accessibility tree |
| Setup | none | `pip install playwright && playwright install chromium` (~300 MB) |

The loop shape does not change, which is the point of learning it here. What changes is the
*failure* surface: real pages move under you, and a screenshot can be stale by the time the
click lands.

**And the practical conclusion:** if a selector or an accessibility tree is available, use
it. Coordinate clicking is the fallback for when nothing else is exposed, not the default.
Production computer-use systems reach for pixels last, not first — hence the hosted
`computer_use_preview` tool in `07_Hosted_vs_Client_Side_Tools.ipynb`, which exists precisely
because doing this well is harder than it looks.

## Key takeaways

1. **Text observations hide the hard part.** Once the screen is pixels, the agent must
   localise controls itself, and that is where computer use actually fails.
2. **Recognising and locating are different skills.** Expect accurate descriptions paired
   with coordinates that are tens of pixels off.
3. **The signature failure is a silent no-op**, not an exception. A missed click changes
   nothing, so a step cap is not optional — it is the only bound on the loop.
4. **Frontend testing is the strongest use case**, because assertions give the loop a
   verifiable end state that open-ended automation lacks.
5. **Log the clicks and what they hit.** Without it you cannot tell a broken UI from a
   confused agent, and every failure gets misattributed.
6. **Prefer selectors to coordinates.** Pixels are the fallback for interfaces that expose
   nothing else.

### Next

- `07_Hosted_vs_Client_Side_Tools.ipynb` — `computer_use_preview` runs this loop on the
  provider's side, with the control trade-offs set out there.